Hava melumatlarini Api ile cekek

şəhərlərin kordinatlarini cities deyisenine menimsedek

In [18]:
import requests
import pandas as pd
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent.parent

cities = {
    "Bakı": (40.4093, 49.8671),
    "Gəncə": (40.6828, 46.3606),
    "Sumqayıt": (40.5897, 49.6686),
    "Lənkəran": (38.7543, 48.8506),
    "Mingəçevir": (40.7703, 47.0489),
    "Şəki": (41.1919, 47.1706),
    "Naxçıvan": (39.2089, 45.4122)
}

In [19]:
print("Cari qovluq:", Path.cwd())
print("PROJECT_ROOT:", PROJECT_ROOT)


Cari qovluq: c:\Users\Sabina\Documents\GitHub\Bank_Analysis\notebooks\1_Bronze
PROJECT_ROOT: c:\Users\Sabina\Documents\GitHub\Bank_Analysis


In [20]:
weather_list = []

for city, (lat, lon) in cities.items():
    params = {
        "latitude": lat,
        "longitude": lon,
        "start_date": "2026-01-01",
        "end_date": "2026-06-30",
        "daily": "temperature_2m_mean,precipitation_sum,wind_speed_10m_max",
        "timezone": "Asia/Baku"
    }

    response = requests.get("https://archive-api.open-meteo.com/v1/archive",
                            params=params, timeout=60)
    response.raise_for_status()

    df = pd.DataFrame(response.json()["daily"])
    df["city"] = city
    weather_list.append(df)

cedvelleri birlesdirek

In [21]:
weather_all_df = pd.concat(weather_list, ignore_index=True)
weather_all_df.shape

(1267, 5)

csv kimi saxlayaq

In [22]:
weather_all_df.to_csv(PROJECT_ROOT / "data/api/weather_7_cities.csv",
                      index=False, encoding="utf-8-sig")

Sql e gonderek

In [25]:
import sys
from sqlalchemy import Unicode

sys.path.insert(0, str(PROJECT_ROOT))

from src.db import get_engine

engine = get_engine()

weather_all_df.to_sql(
    "weather",
    engine,
    schema="bronze",
    if_exists="replace",
    index=False,
    dtype={"city": Unicode(100)})

10

Teqvim melumatlarini sql e gonderek


In [26]:
calendar_df = pd.read_csv(
    PROJECT_ROOT / "data" / "bank_data" / "calendar_reference.csv"
)

print("Sətir və sütun sayı:", calendar_df.shape)
display(calendar_df.head())

Sətir və sütun sayı: (181, 6)


,date,is_holiday,holiday_name,is_weekend,is_workday,is_pre_holiday
0,1/1/2026,1,Yeni il,0,0,0
1,1/2/2026,1,Yeni il,0,0,0
2,1/3/2026,0,NaN,1,0,0
3,1/4/2026,0,NaN,1,0,0
4,1/5/2026,0,NaN,0,1,0


In [28]:
from sqlalchemy import text, Unicode
import sys

sys.path.insert(0, str(PROJECT_ROOT))
from src.db import get_engine

engine = get_engine()

calendar_df.to_sql(
    name="calendar_reference",
    con=engine,
    schema="bronze",
    if_exists="replace",
    index=False,
    dtype={c: Unicode(255) for c in calendar_df.select_dtypes(include=["string", "object"]).columns})

with engine.connect() as conn:
    count = conn.execute(
        text("SELECT COUNT(*) FROM bronze.calendar_reference")
    ).scalar()

print("bronze.calendar_reference:", count, "sətir")

bronze.calendar_reference: 181 sətir


In [29]:
pd.read_sql("SELECT DISTINCT holiday_name FROM bronze.calendar_reference WHERE holiday_name IS NOT NULL", engine)

,holiday_name
0,Əlavə istirahət günü
1,Milli Qurtuluş Günü
2,Novruz
3,Novruz / Ramazan bayramı
4,Qadınlar günü
5,Qadınlar günü (əvəz istirahət)
6,Qələbə Günü
7,Qələbə Günü (əvəz istirahət)
8,Qurban bayramı
9,Qurban bayramı / Müstəqillik Günü


Indi məzənnə məlumatlarını api ilə çəkək

In [30]:
import requests
import pandas as pd

url = "https://api.frankfurter.dev/v2/rates"

params = {
    "from": "2026-01-01",
    "to": "2026-06-30",
    "base": "AZN",
    "quotes": "USD,EUR"
}

response = requests.get(url, params=params, timeout=60)

print("HTTP status:", response.status_code)

if response.ok:
    fx_data = response.json()
    fx_df = pd.DataFrame(fx_data)

    print("Sətir və sütun sayı:", fx_df.shape)
    display(fx_df.head())
else:
    print("API cavabı:", response.text[:500])

HTTP status: 200
Sətir və sütun sayı: (362, 4)


,date,base,quote,rate
0,2026-01-01,AZN,EUR,0.50084
1,2026-01-01,AZN,USD,0.58837
2,2026-01-02,AZN,EUR,0.50095
3,2026-01-02,AZN,USD,0.58830
4,2026-01-03,AZN,EUR,0.50107


csv kimi api qovlugunda saxlayaq

In [31]:
from pathlib import Path

API_DIR = PROJECT_ROOT / "data" / "api"
API_DIR.mkdir(parents=True, exist_ok=True)

file_path = API_DIR / "exchange_rates_2026_01_06.csv"

fx_df.to_csv(file_path, index=False, encoding="utf-8")

print("Fayl saxlanıldı:", file_path)
print("Sətir sayı:", len(fx_df))

Fayl saxlanıldı: c:\Users\Sabina\Documents\GitHub\Bank_Analysis\data\api\exchange_rates_2026_01_06.csv
Sətir sayı: 362


sql servere gonderek

In [33]:
import sys
from sqlalchemy import text

sys.path.insert(0, str(PROJECT_ROOT))
from src.db import get_engine

engine = get_engine()

fx_df.to_sql(
    name="exchange_rates",
    con=engine,
    schema="bronze",
    if_exists="replace",
    index=False
)

with engine.connect() as conn:
    count = conn.execute(
        text("SELECT COUNT(*) FROM bronze.exchange_rates")
    ).scalar()

print("bronze.exchange_rates:", count, "sətir")

bronze.exchange_rates: 362 sətir


In [34]:
from sqlalchemy import text

with engine.connect() as conn:
    result = conn.execute(text("""
        SELECT
            TABLE_SCHEMA,
            TABLE_NAME
        FROM INFORMATION_SCHEMA.TABLES
        WHERE TABLE_SCHEMA = 'bronze'
        ORDER BY TABLE_NAME
    """))

    for row in result:
        print(f"{row.TABLE_SCHEMA}.{row.TABLE_NAME}")

bronze.calendar_reference
bronze.customers
bronze.exchange_rates
bronze.loans
bronze.loans_enriched
bronze.transactions
bronze.transactions_enriched
bronze.weather
